## MapType() Column
The `MapType()` in PySpark defines a column type for data that is a map of key-value pairs. It is commonly used when your data contains dictionaries or JSON objects. The syntax is:

python
from pyspark.sql.types import MapType, StringType, IntegerType

### Example: a column with string keys and integer values
MapType(StringType(), IntegerType())


This allows you to store and query complex, nested data structures within a DataFrame.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
data = [('kanth',{'age':19,'height':5.11}),('raj',{'age':20,'height':5.12})]
schema = StructType([\
    StructField('name',StringType()),\
    StructField('details',MapType(StringType(),DoubleType()))])
df = spark.createDataFrame(data,schema)
df.show(truncate = False)
df.printSchema()

+-----+-----------------------------+
|name |details                      |
+-----+-----------------------------+
|kanth|{age -> 19.0, height -> 5.11}|
|raj  |{age -> 20.0, height -> 5.12}|
+-----+-----------------------------+

root
 |-- name: string (nullable = true)
 |-- details: map (nullable = true)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)



### Accessing MapType Elements

You can access elements in a `MapType` column using the `getItem` method or bracket notation. For example, to select the `age` and `height` from the `details` map:

python
df.select(
    "name",
    df.details.getItem("age"),
    df.details["height"]
).show()

if we want new columns

In [0]:
df1 = df.withColumn('age',df.details['age'])
df1.show(truncate = False)
df2 = df1.withColumn('height',df1.details.getItem('height'))
df2.show(truncate = False)

+-----+-----------------------------+----+
|name |details                      |age |
+-----+-----------------------------+----+
|kanth|{age -> 19.0, height -> 5.11}|19.0|
|raj  |{age -> 20.0, height -> 5.12}|20.0|
+-----+-----------------------------+----+

+-----+-----------------------------+----+------+
|name |details                      |age |height|
+-----+-----------------------------+----+------+
|kanth|{age -> 19.0, height -> 5.11}|19.0|5.11  |
|raj  |{age -> 20.0, height -> 5.12}|20.0|5.12  |
+-----+-----------------------------+----+------+



In [0]:
df3 = df.select("name","details",explode("details"))
df3.show(truncate = False)

+-----+-----------------------------+------+-----+
|name |details                      |key   |value|
+-----+-----------------------------+------+-----+
|kanth|{age -> 19.0, height -> 5.11}|age   |19.0 |
|kanth|{age -> 19.0, height -> 5.11}|height|5.11 |
|raj  |{age -> 20.0, height -> 5.12}|age   |20.0 |
|raj  |{age -> 20.0, height -> 5.12}|height|5.12 |
+-----+-----------------------------+------+-----+



## MapKeys() and MapValues()
The `map_keys()` and `map_values()` functions in PySpark are used to extract all the keys or all the values from a `MapType` column as arrays.

**Example:**

python
from pyspark.sql.functions import map_keys, map_values

df.select(
    "name",
    map_keys("details").alias("keys"),
    map_values("details").alias("values")
).show(truncate=False)


This will display the keys and values from the `details` map for each row.

In [0]:
df4 = df.withColumn('keys',map_keys(df.details))
df4.show(truncate = False)

+-----+-----------------------------+-------------+
|name |details                      |keys         |
+-----+-----------------------------+-------------+
|kanth|{age -> 19.0, height -> 5.11}|[age, height]|
|raj  |{age -> 20.0, height -> 5.12}|[age, height]|
+-----+-----------------------------+-------------+



In [0]:
df5 = df4.withColumn('values',map_values(df.details))
df5.show(truncate = False)

+-----+-----------------------------+-------------+------------+
|name |details                      |keys         |values      |
+-----+-----------------------------+-------------+------------+
|kanth|{age -> 19.0, height -> 5.11}|[age, height]|[19.0, 5.11]|
|raj  |{age -> 20.0, height -> 5.12}|[age, height]|[20.0, 5.12]|
+-----+-----------------------------+-------------+------------+

